# Iteration 04 — Fine-tune LLaMA-3.2-1B on one arm + Bengali evaluation

One run = one arm × one seed. Run it for arms **A, B, C** first (then D, E, F if compute allows), ideally with
3 seeds each; every run appends one row to `results.csv` and saves per-item scores, so the last section can
compare arms with paired bootstrap confidence intervals.

| Setting | Value | Source |
|---|---|---|
| Base model | `meta-llama/Llama-3.2-1B` (raw base; TigerLLM also continued-pretrained on Bangla-TextBook) | Task 3 §3.1 |
| Recipe (`MODE="full"`) | full FT, 3 epochs, lr 1e-5, wd 0.02, 10% warm-up, cosine, AdamW-8bit, BF16, seq 2048, effective batch 64 | TigerLLM |
| Pilot (`MODE="pilot"`) | LoRA r=16, 300 steps — only checks that code, data format and hardware work | Task 4 step 5 |
| Loss | on response tokens only (prompt tokens masked) | |
| Template | `### নির্দেশনা:\n…\n\n### উত্তর:\n…<eos>` — same for all arms and for evaluation | |

**Evaluation** (all arms, same prompts, greedy decoding):
* **MMLU-bn** — `openai/MMMLU` BN-BD (14,042 questions), zero-shot, next-token letter scoring
* **BanglaRQA** — 1,493 test questions (incl. unanswerable), EM / token-F1
* **mHumanEval-bn** — 164 Python tasks with Bengali docstrings, pass@1 (executes generated code: Colab only)
* **held-out loss** on the 500 held-out Bangla-Instruct prompts (never in any arm)
* *(optional)* pairwise LLM-judge win rate on held-out prompts

BEnQA, BanglaQuaD and PangBench-bn are not on the HF Hub under those names; once you have the files,
`eval_mcq()` / `eval_extractive_qa()` below take any dataframe. Our MMLU-bn may not be the exact split
TigerLLM used, so compare **arms with each other**, not against TigerLLM's table.

In [ ]:
%pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets huggingface_hub pandas scipy matplotlib

In [ ]:
import os, re, sys, json, math, time, random, string, subprocess, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# --- platform: Kaggle / Colab / local ----------------------------------------------------------------
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = os.path.exists("/content") and not ON_KAGGLE

def hf_login():
    """Read HF_TOKEN from Kaggle Secrets / Colab secrets (needed for the gated meta-llama weights)."""
    token = os.environ.get("HF_TOKEN")
    try:
        if not token and ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        elif not token and ON_COLAB:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN secret found:", type(e).__name__)
    if token:
        os.environ["HF_TOKEN"] = token
        from huggingface_hub import login
        login(token=token)

hf_login()

CONFIG = {
    "ARM":   "C",            # A | B | C | D | E | F | base  ("base" = no fine-tuning, evaluation only)
    "SEED":  42,             # run 42, 43, 44 for the final comparison
    "MODE":  "pilot",        # "pilot" | "full"
    # Kaggle: publish iteration03's arms/ folder as a Kaggle Dataset and attach it, e.g. /kaggle/input/bangla-arms
    "ARMS_DIR": "/kaggle/input/bangla-arms" if ON_KAGGLE else "/content/bangla_filtering/full/arms",
    "BASE_MODEL":          "meta-llama/Llama-3.2-1B",
    "BASE_MODEL_FALLBACK": "unsloth/Llama-3.2-1B",
    "OUT_DIR":  "/kaggle/working/bangla_runs" if ON_KAGGLE else "/content/bangla_runs",
    "USE_DRIVE": False,

    # TigerLLM 1B recipe
    "MAX_LEN": 2048, "EPOCHS": 3, "LR": 1e-5, "WEIGHT_DECAY": 0.02, "WARMUP_FRAC": 0.10,
    "EFFECTIVE_BATCH": 64,   # = batch 16 × grad-accum 4
    "PER_DEVICE_BS": 16,     # lowered automatically on smaller GPUs; EFFECTIVE_BATCH stays 64
    "OPTIM": "adamw_bnb_8bit",

    # pilot
    "PILOT_STEPS": 300, "PILOT_LR": 2e-4, "PILOT_BS": 4, "PILOT_MAX_LEN": 1024,
    "LORA_R": 16, "LORA_ALPHA": 32,

    # evaluation sizes (None = everything; pilot uses small samples)
    "EVAL_MMLU_N": None, "EVAL_RQA_N": None, "EVAL_MHE_N": None, "EVAL_BS": 16,
    "MHUMANEVAL_BN_URL": "https://raw.githubusercontent.com/mraihan-gmu/mHumanEval-Benchmark/main/mHumanEval-max/mHumanEval-ben_Beng-python.csv",
}
if CONFIG["MODE"] == "pilot":
    CONFIG.update(EVAL_MMLU_N=500, EVAL_RQA_N=200, EVAL_MHE_N=40)
if CONFIG["USE_DRIVE"] and ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CONFIG["OUT_DIR"] = "/content/drive/MyDrive/bangla_runs"

if ON_KAGGLE and not Path(CONFIG["ARMS_DIR"], "heldout.jsonl").exists():
    # arms attached as a kernel source (iteration03's output) or a dataset: find them wherever they are mounted
    hits = sorted(Path("/kaggle/input").glob("**/arms/heldout.jsonl"))
    if hits:
        CONFIG["ARMS_DIR"] = str(hits[-1].parent)
print("arms:", CONFIG["ARMS_DIR"])

RUN_NAME = f"{CONFIG['ARM']}_{CONFIG['MODE']}_s{CONFIG['SEED']}"
RUN_DIR = Path(CONFIG["OUT_DIR"]) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
ARMS_DIR = Path(CONFIG["ARMS_DIR"])

random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"]); torch.manual_seed(CONFIG["SEED"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else (torch.float16 if DEVICE == "cuda" else torch.float32)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE == "cuda" else 0
print(RUN_NAME, "|", DEVICE, f"{GPU_GB:.0f} GB" if GPU_GB else "", "| bf16:", BF16)

PROMPT_TMPL = "### নির্দেশনা:\n{instruction}\n\n### উত্তর:\n"     # identical to iteration03

---
## 1. Tokenizer, data, token budget

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_tokenizer():
    for name in (CONFIG["BASE_MODEL"], CONFIG["BASE_MODEL_FALLBACK"]):
        try:
            return name, AutoTokenizer.from_pretrained(name)
        except Exception as e:
            print(f"cannot load {name}: {type(e).__name__} (gated? set HF_TOKEN and accept the licence)")
    raise RuntimeError("no base model available")

BASE_NAME, tok = load_tokenizer()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.truncation_side = "left"      # long eval prompts lose the start of the passage, never the answer header
BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

def encode(instruction, output, max_len):
    p = BOS + tok(PROMPT_TMPL.format(instruction=instruction), add_special_tokens=False)["input_ids"]
    r = tok(output, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(p) > max_len - 32:                       # prompt alone does not fit
        return None
    r = r[: max_len - len(p)]
    return {"input_ids": p + r, "labels": [-100] * len(p) + r}

MAX_LEN = CONFIG["PILOT_MAX_LEN"] if CONFIG["MODE"] == "pilot" else CONFIG["MAX_LEN"]
held_rows = read_jsonl(ARMS_DIR / "heldout.jsonl")
eval_feats = [e for e in (encode(r["instruction"], r["output"], MAX_LEN) for r in held_rows) if e]

if CONFIG["ARM"] != "base":
    train_rows = read_jsonl(ARMS_DIR / f"arm_{CONFIG['ARM']}.jsonl")
    train_feats = [encode(r["instruction"], r["output"], MAX_LEN) for r in tqdm(train_rows, desc="tokenize")]
    n_skipped = sum(f is None for f in train_feats)
    train_feats = [f for f in train_feats if f]
    n_trunc = sum(f["labels"][-1] != tok.eos_token_id for f in train_feats)
    TOKENS = {"examples": len(train_feats), "skipped_prompt_too_long": n_skipped, "response_truncated": n_trunc,
              "tokens_total": sum(len(f["input_ids"]) for f in train_feats),
              "tokens_supervised": sum(sum(l != -100 for l in f["labels"]) for f in train_feats)}
    print(json.dumps(TOKENS, indent=2))     # report next to the scores: arms differ in tokens, not only rows

---
## 2. Train

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, feats): self.feats = feats
    def __len__(self): return len(self.feats)
    def __getitem__(self, i): return self.feats[i]

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

if CONFIG["ARM"] == "base":
    model = AutoModelForCausalLM.from_pretrained(BASE_NAME, torch_dtype=DTYPE).to(DEVICE)
    TRAIN_META = {"train_runtime_s": 0}
else:
    pilot = CONFIG["MODE"] == "pilot"
    if pilot:
        from peft import LoraConfig, get_peft_model
        model = AutoModelForCausalLM.from_pretrained(BASE_NAME, torch_dtype=DTYPE)
        model = get_peft_model(model, LoraConfig(
            r=CONFIG["LORA_R"], lora_alpha=CONFIG["LORA_ALPHA"], lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
        model.print_trainable_parameters()
        pdbs, ga = CONFIG["PILOT_BS"], 4
    else:
        model = AutoModelForCausalLM.from_pretrained(BASE_NAME, torch_dtype=torch.float32)  # fp32 master weights + bf16 autocast
        pdbs = CONFIG["PER_DEVICE_BS"] if GPU_GB >= 70 else (8 if GPU_GB >= 38 else (4 if GPU_GB >= 22 else 1))
    n_gpu = max(1, torch.cuda.device_count())
    if not pilot:
        ga = max(1, CONFIG["EFFECTIVE_BATCH"] // (pdbs * n_gpu))     # 2×T4 on Kaggle: Trainer uses both GPUs
    model.config.use_cache = False

    steps_per_epoch = math.ceil(len(train_feats) / (pdbs * ga * n_gpu))
    total_steps = CONFIG["PILOT_STEPS"] if pilot else steps_per_epoch * CONFIG["EPOCHS"]
    args = TrainingArguments(
        output_dir=str(RUN_DIR / "ckpt"), seed=CONFIG["SEED"], data_seed=CONFIG["SEED"],
        per_device_train_batch_size=pdbs, gradient_accumulation_steps=ga, per_device_eval_batch_size=pdbs,
        learning_rate=CONFIG["PILOT_LR"] if pilot else CONFIG["LR"],
        weight_decay=CONFIG["WEIGHT_DECAY"], lr_scheduler_type="cosine",
        warmup_steps=int(CONFIG["WARMUP_FRAC"] * total_steps),
        num_train_epochs=CONFIG["EPOCHS"], max_steps=total_steps if pilot else -1,
        optim="adamw_torch" if pilot else CONFIG["OPTIM"],
        bf16=BF16, fp16=(DEVICE == "cuda" and not BF16),
        gradient_checkpointing=DEVICE == "cuda", gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10, eval_strategy="steps", eval_steps=max(10, total_steps // 20),
        save_strategy="steps", save_steps=max(50, total_steps // 6), save_total_limit=1,
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2 if DEVICE == "cuda" else 0,
    )
    print(f"per-device batch {pdbs} × {n_gpu} GPU × grad-accum {ga} = {pdbs * ga * n_gpu} | steps {total_steps:,}")
    trainer = Trainer(model=model, args=args, data_collator=collate,
                      train_dataset=ListDataset(train_feats), eval_dataset=ListDataset(eval_feats))
    last = get_last_checkpoint(args.output_dir) if Path(args.output_dir).exists() else None
    t0 = time.time()
    trainer.train(resume_from_checkpoint=last)       # resumes after a Colab disconnect
    TRAIN_META = {"train_runtime_s": round(time.time() - t0), "per_device_bs": pdbs, "grad_accum": ga,
                  "steps": total_steps, **TOKENS}

    hist = pd.DataFrame(trainer.state.log_history)
    hist.to_csv(RUN_DIR / "log_history.csv", index=False)
    if pilot:
        model = model.merge_and_unload()
    model.save_pretrained(RUN_DIR / "model"); tok.save_pretrained(RUN_DIR / "model")
    model = model.to(DTYPE)

    import matplotlib.pyplot as plt
    ax = hist.dropna(subset=["loss"]).plot(x="step", y="loss", figsize=(8, 3), label="train")
    if "eval_loss" in hist:
        hist.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="held-out")
    plt.title(RUN_NAME); plt.savefig(RUN_DIR / "loss.png", dpi=110); plt.show()

model.eval(); model.config.use_cache = True

---
## 3. Evaluation

In [ ]:
from huggingface_hub import hf_hub_download

def wrap(instruction):
    return (tok.bos_token or "") + PROMPT_TMPL.format(instruction=instruction)

@torch.no_grad()
def generate(instructions, max_new_tokens=256, bs=None):
    bs = bs or CONFIG["EVAL_BS"]
    tok.padding_side = "left"
    order = np.argsort([-len(x) for x in instructions])
    outs = [None] * len(instructions)
    for b in tqdm(range(0, len(order), bs), desc="generate", leave=False):
        idx = order[b:b + bs]
        enc = tok([wrap(instructions[i]) for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=False, truncation=True, max_length=CONFIG["MAX_LEN"] - max_new_tokens).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for i, g in zip(idx, tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            outs[i] = g.split("### নির্দেশনা:")[0].strip()
    tok.padding_side = "right"
    return outs

def option_token_ids(labels):
    """ids of 'A' and ' A' (etc.) — summed, so the score does not depend on how the tokenizer splits the space."""
    return [sorted({tok.encode(l, add_special_tokens=False)[0], tok.encode(" " + l, add_special_tokens=False)[-1]})
            for l in labels]

@torch.no_grad()
def eval_mcq(frame, question_col, option_cols, answer_col, labels=("A", "B", "C", "D"), name="mcq"):
    """Zero-shot MCQ by next-token probability of the option letter. Works for any dataframe."""
    ids = option_token_ids(labels)
    prompts = [("নিচের বহুনির্বাচনী প্রশ্নের সঠিক উত্তর বেছে নিন। শুধু অক্ষরটি (" + ", ".join(labels) + ") লিখুন।\n\n"
                f"প্রশ্ন: {r[question_col]}\n" + "\n".join(f"{l}. {r[c]}" for l, c in zip(labels, option_cols)))
               for _, r in frame.iterrows()]
    tok.padding_side = "left"
    preds = []
    for b in tqdm(range(0, len(prompts), CONFIG["EVAL_BS"]), desc=name, leave=False):
        enc = tok([wrap(p) for p in prompts[b:b + CONFIG["EVAL_BS"]]], return_tensors="pt", padding=True,
                  add_special_tokens=False, truncation=True, max_length=CONFIG["MAX_LEN"]).to(DEVICE)
        probs = model(**enc).logits[:, -1].float().softmax(-1)
        scores = torch.stack([probs[:, i].sum(-1) for i in ids], dim=1)
        preds += [labels[k] for k in scores.argmax(1).tolist()]
    tok.padding_side = "right"
    correct = (np.array(preds) == frame[answer_col].astype(str).str.strip().values).astype(int)
    return correct, preds

PER_ITEM, RESULTS = {}, {}

# --- MMLU-bn -------------------------------------------------------------------------------------------
mmlu = pd.read_csv(hf_hub_download("openai/MMMLU", "test/mmlu_BN-BD.csv", repo_type="dataset"))
if CONFIG["EVAL_MMLU_N"]:
    mmlu = mmlu.sample(CONFIG["EVAL_MMLU_N"], random_state=0)        # same items for every run
mmlu = mmlu.sort_index()
correct, _ = eval_mcq(mmlu, "Question", ["A", "B", "C", "D"], "Answer", name="MMLU-bn")
PER_ITEM["mmlu_bn"] = dict(zip(mmlu.index.astype(str), correct.tolist()))
RESULTS["mmlu_bn_acc"] = round(float(correct.mean()), 4)
print("MMLU-bn accuracy:", RESULTS["mmlu_bn_acc"], "(chance = 0.25)")
display(pd.Series(correct, index=mmlu["Subject"].values).groupby(level=0).mean().sort_values().round(3).head(10))

In [ ]:
# --- BanglaRQA (extractive / yes-no / unanswerable QA) --------------------------------------------------
NO_ANSWER = "উত্তর দেওয়া সম্ভব নয়"
PUNCT = set(string.punctuation) | set("।॥‘’“”…–—")

def norm_answer(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = "".join(ch for ch in s if ch not in PUNCT)
    return " ".join(s.split())

def token_f1(pred, gold):
    p, g = norm_answer(pred).split(), norm_answer(gold).split()
    common = sum(min(p.count(t), g.count(t)) for t in set(p))
    if not p or not g or common == 0:
        return float(p == g)
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

def eval_extractive_qa(items, name="qa"):
    """items: list of dicts with context, question, golds (list[str]), answerable (bool)."""
    prompts = [("নিচের অনুচ্ছেদটি পড়ে প্রশ্নের উত্তর দিন। উত্তর অনুচ্ছেদ থেকে সংক্ষেপে লিখুন। "
                f"অনুচ্ছেদে উত্তর না থাকলে লিখুন: \"{NO_ANSWER}\"।\n\nঅনুচ্ছেদ: {it['context'][:3000]}\n\nপ্রশ্ন: {it['question']}")
               for it in items]
    preds = [p.split("\n")[0].strip() for p in generate(prompts, max_new_tokens=64)]
    em, f1 = [], []
    for it, p in zip(items, preds):
        said_none = "সম্ভব নয়" in p or "উত্তর নেই" in p
        if not it["answerable"]:
            em.append(int(said_none)); f1.append(float(said_none)); continue
        if said_none or not it["golds"]:
            em.append(0); f1.append(0.0); continue
        em.append(int(max(norm_answer(p) == norm_answer(g) for g in it["golds"])))
        f1.append(max(token_f1(p, g) for g in it["golds"]))
    return np.array(em), np.array(f1), preds

rqa = json.load(open(hf_hub_download("sartajekram/BanglaRQA", "Test.json", repo_type="dataset"), encoding="utf-8"))["data"]
rqa_items = [{"id": q["question_id"], "context": p["context"], "question": q["question_text"],
              "golds": [a for a in q["answers"]["answer_text"] if a], "answerable": str(q["is_answerable"]) == "1",
              "type": q.get("question_type", "")} for p in rqa for q in p["qas"]]
if CONFIG["EVAL_RQA_N"]:
    rqa_items = random.Random(0).sample(rqa_items, CONFIG["EVAL_RQA_N"])
em, f1, rqa_preds = eval_extractive_qa(rqa_items, "BanglaRQA")
PER_ITEM["banglarqa_f1"] = {it["id"]: float(v) for it, v in zip(rqa_items, f1)}
RESULTS.update(banglarqa_em=round(float(em.mean()), 4), banglarqa_f1=round(float(f1.mean()), 4))
print("BanglaRQA EM / F1:", RESULTS["banglarqa_em"], RESULTS["banglarqa_f1"])
display(pd.DataFrame({"type": [it["type"] for it in rqa_items], "f1": f1}).groupby("type")["f1"].agg(["mean", "size"]).round(3))

In [ ]:
# --- mHumanEval-bn (Python, pass@1) --------------------------------------------------------------------
# ⚠ executes model-generated code in a subprocess. Run this on Colab / a throwaway VM, never on your laptop.
CODE_BLOCK = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)

def run_program(program, timeout=10):
    try:
        r = subprocess.run([sys.executable, "-c", program], capture_output=True, timeout=timeout, text=True)
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False

mhe = pd.read_csv(CONFIG["MHUMANEVAL_BN_URL"])[["task_id", "prompt", "test"]]
if CONFIG["EVAL_MHE_N"]:
    mhe = mhe.head(CONFIG["EVAL_MHE_N"])
mhe_prompts = ["নিচের Python ফাংশনটি সম্পূর্ণ করুন। পুরো ফাংশনটি একটি ```python কোড ব্লকে লিখুন।\n\n```python\n"
               + p.rstrip() + "\n```" for p in mhe["prompt"]]
mhe_out = generate(mhe_prompts, max_new_tokens=512, bs=8)
passed = []
for (_, r), out in zip(mhe.iterrows(), mhe_out):
    m = CODE_BLOCK.search(out + "\n```")
    code = m.group(1) if m else out
    entry = re.findall(r"def\s+(\w+)\s*\(", r["prompt"])[-1]
    # prompt (stub with docstring) + completion: works whether the model wrote the body or the whole def
    program = r["prompt"].rstrip() + "\n" + code + "\n\n" + r["test"] + f"\n\ncheck({entry})\n"
    passed.append(int(run_program(program)))
PER_ITEM["mhumaneval_bn"] = dict(zip(mhe["task_id"], passed))
RESULTS["mhumaneval_bn_pass1"] = round(float(np.mean(passed)), 4)
print("mHumanEval-bn pass@1:", RESULTS["mhumaneval_bn_pass1"])

In [ ]:
# --- held-out loss + held-out generations (inputs for the optional win-rate comparison) -----------------
@torch.no_grad()
def mean_loss(feats, bs=8):
    tot, n = 0.0, 0
    for b in range(0, len(feats), bs):
        batch = {k: v.to(DEVICE) for k, v in collate(feats[b:b + bs]).items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits[:, :-1].float()
        lab = batch["labels"][:, 1:]
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), lab.reshape(-1),
                                                 ignore_index=-100, reduction="sum")
        tot += loss.item(); n += int((lab != -100).sum())
    return tot / n

RESULTS["heldout_loss"] = round(mean_loss(eval_feats), 4)
held_gen = generate([r["instruction"] for r in held_rows], max_new_tokens=512)
with open(RUN_DIR / "heldout_generations.jsonl", "w", encoding="utf-8") as f:
    for r, g in zip(held_rows, held_gen):
        f.write(json.dumps({"id": r["id"], "instruction": r["instruction"], "generation": g}, ensure_ascii=False) + "\n")
print("held-out loss:", RESULTS["heldout_loss"])
for r, g in list(zip(held_rows, held_gen))[:2]:
    print("\nQ:", r["instruction"][:200], "\nA:", g[:300])

In [ ]:
row = {"run": RUN_NAME, "arm": CONFIG["ARM"], "mode": CONFIG["MODE"], "seed": CONFIG["SEED"],
       "base_model": BASE_NAME, **RESULTS, **TRAIN_META}
(RUN_DIR / "per_item.json").write_text(json.dumps(PER_ITEM, ensure_ascii=False), encoding="utf-8")
(RUN_DIR / "result.json").write_text(json.dumps(row, indent=2, ensure_ascii=False), encoding="utf-8")
res_path = Path(CONFIG["OUT_DIR"]) / "results.csv"
all_res = pd.concat([pd.read_csv(res_path), pd.DataFrame([row])]) if res_path.exists() else pd.DataFrame([row])
all_res.drop_duplicates("run", keep="last").to_csv(res_path, index=False)
print(json.dumps(row, indent=2, ensure_ascii=False))

---
## 4. Compare arms (run after several arms/seeds are done)
Paired bootstrap over benchmark items: for each seed run in both arms, resample items, average the difference.
`P(Δ ≤ 0)` is the share of resamples where the first arm is *not* better (≈ a one-sided p-value).

In [ ]:
def load_per_item(arm, mode="full"):
    runs = sorted(Path(CONFIG["OUT_DIR"]).glob(f"{arm}_{mode}_s*/per_item.json"))
    return [json.loads(p.read_text(encoding="utf-8")) for p in runs]

def paired_bootstrap(arm_x, arm_y, bench, mode="full", n_boot=10_000):
    X, Y = load_per_item(arm_x, mode), load_per_item(arm_y, mode)
    if not X or not Y:
        return None
    keys = sorted(set.intersection(*[set(r[bench]) for r in X + Y]))
    x = np.mean([[r[bench][k] for k in keys] for r in X], axis=0)     # seed-averaged per-item score
    y = np.mean([[r[bench][k] for k in keys] for r in Y], axis=0)
    idx = np.random.default_rng(0).integers(0, len(keys), (n_boot, len(keys)))
    d = x[idx].mean(1) - y[idx].mean(1)
    return {"pair": f"{arm_x} vs {arm_y}", "bench": bench, "n_items": len(keys), "seeds": (len(X), len(Y)),
            "delta": round(float(x.mean() - y.mean()), 4),
            "ci95": [round(float(v), 4) for v in np.quantile(d, [0.025, 0.975])],
            "P(delta<=0)": round(float((d <= 0).mean()), 4)}

res_path = Path(CONFIG["OUT_DIR"]) / "results.csv"
if res_path.exists():
    res = pd.read_csv(res_path)
    metrics = [c for c in ["mmlu_bn_acc", "banglarqa_f1", "mhumaneval_bn_pass1", "heldout_loss", "tokens_total"] if c in res]
    display(res.groupby(["mode", "arm"])[metrics].agg(["mean", "std", "count"]).round(4))
    comps = [paired_bootstrap(a, b, bench, CONFIG["MODE"])
             for a, b in [("C", "B"), ("C", "A"), ("D", "C"), ("E", "C")]
             for bench in ["mmlu_bn", "banglarqa_f1", "mhumaneval_bn"]]
    display(pd.DataFrame([c for c in comps if c]))

### Optional — pairwise LLM-judge win rate on the held-out prompts
Both orders are judged (swap) to cancel position bias; the score is P("1") vs P("2") on the judge's first token.
Run in a fresh runtime after the generations of both arms exist.

In [ ]:
def pairwise_winrate(run_x, run_y, judge_name="Qwen/Qwen2.5-7B-Instruct", n=200, bs=4):
    jt = AutoTokenizer.from_pretrained(judge_name, padding_side="left")
    jt.pad_token = jt.pad_token or jt.eos_token
    jm = AutoModelForCausalLM.from_pretrained(judge_name, torch_dtype=DTYPE, device_map="auto").eval()
    one, two = jt.encode("1", add_special_tokens=False)[-1], jt.encode("2", add_special_tokens=False)[-1]
    read = lambda run: {r["id"]: r for r in read_jsonl(Path(CONFIG["OUT_DIR"]) / run / "heldout_generations.jsonl")}
    X, Y = read(run_x), read(run_y)
    ids = sorted(set(X) & set(Y))[:n]
    tmpl = ("Two assistants answered the same Bengali instruction. Which response is better overall "
            "(correct, complete, helpful, natural Bengali)?\n\nInstruction:\n{q}\n\nResponse 1:\n{a}\n\nResponse 2:\n{b}\n\n"
            "Reply with 1 or 2.")
    def p_first(pairs):
        out = []
        for b in range(0, len(pairs), bs):
            msgs = [jt.apply_chat_template([{"role": "user", "content": tmpl.format(q=q[:1500], a=a[:1500], b=c[:1500])}],
                                           tokenize=False, add_generation_prompt=True) for q, a, c in pairs[b:b + bs]]
            enc = jt(msgs, return_tensors="pt", padding=True, add_special_tokens=False).to(jm.device)
            with torch.no_grad():
                pr = jm(**enc).logits[:, -1].float().softmax(-1)
            out += (pr[:, one] / (pr[:, one] + pr[:, two])).tolist()
        return np.array(out)
    fwd = p_first([(X[i]["instruction"], X[i]["generation"], Y[i]["generation"]) for i in ids])
    rev = p_first([(X[i]["instruction"], Y[i]["generation"], X[i]["generation"]) for i in ids])
    win = (fwd + (1 - rev)) / 2                                         # P(x better), order-averaged
    return {"pair": f"{run_x} vs {run_y}", "n": len(ids), "win_rate_x": round(float((win > 0.5).mean()), 4),
            "mean_pref_x": round(float(win.mean()), 4)}

# pairwise_winrate("C_full_s42", "B_full_s42")